In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.lines import Line2D
import os
import pathlib

In [4]:
import re
import sys
from pathlib import Path

In [5]:
import fast_hdbscan

In [6]:
def parse_phenotype(pheno: str) -> dict[str, str]:
    """Return {marker: '+'/'-'} for a phenotype string like 'CD15-CK+CD3-'."""
    return dict(re.findall(r"([A-Za-z0-9]+)([+\-])", pheno))  # dict() zamiast comprehension


def canonical(pheno: str) -> str:
    """Canonical phenotype: markers sorted alphabetically, signs attached."""
    return "".join(
        f"{m}{s}" for m, s in sorted(re.findall(r"([A-Za-z0-9]+)([+\-])", pheno))
    )  # pomiń parse_phenotype — sortowanie par (m, s) daje ten sam efekt co sortowanie dict


def build_mapping(mapping_file: Path) -> dict[str, str]:
    df = pd.read_csv(mapping_file, usecols=["phenotype", "celltype"])  # wczytaj tylko potrzebne kolumny
    return dict(zip(df["phenotype"].map(canonical), df["celltype"]))   # iterrows() → zip (10-100x szybciej)

In [7]:
def assign_data_to_squares(df: pd.DataFrame, square_a: float = 250) -> pd.DataFrame:
    """
    Zamiast listy list tablic, zwraca df z kolumnami 'square_x' i 'square_y'.
    Przypisanie odbywa się jedną wektoryzowaną operacją — bez pętli.
    """
    x_min = df["nucleus.x"].min()
    y_min = df["nucleus.y"].min()

    df = df.copy()
    df["square_x"] = ((df["nucleus.x"] - x_min) / square_a).astype(int)
    df["square_y"] = ((df["nucleus.y"] - y_min) / square_a).astype(int)

    return df


def calculate_squares_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Przyjmuje df z kolumnami 'square_x', 'square_y' (wynik assign_data_to_squares_fast).
    Używa groupby zamiast iterowania po kwadratach.
    """

    counts = (
        df.groupby(["square_x", "square_y", "cell.type"])
        .size()
        .unstack(fill_value=0)  # cell.type -> kolumny
    )
    counts.columns.name = None

    totals = counts.sum(axis=1)

    proportions = counts.div(totals, axis=0)
    proportions["total"] = totals

    return proportions.reset_index()


def analise_patient(
        df: pd.DataFrame, 
        mapping: dict[str, str],
        square_size: float = 250,
    ) -> pd.DataFrame:
    if "phenotype" not in df.columns:
        raise ValueError(f"{"phenotype"} column not in df!")

    df["cell.type"] = df["phenotype"].map(lambda p: mapping.get(canonical(p), "unknown"))


    return calculate_squares_stats(assign_data_to_squares(df, square_size))    

In [13]:
def plot_scatter_col(df: pd.DataFrame, ax: Axes) -> None:
    legend_points: list[Line2D] = []


    for i, cell_name in enumerate(df["cell.type"].unique()):
        ax.scatter(
            df[df["cell.type"] == cell_name]["nucleus.x"],
            df[df["cell.type"] == cell_name]["nucleus.y"],
            alpha=1,
            marker='.',
            s=0.005,
            label = f"{cell_name}",
            color = f"C{i}",
        )
        legend_points.append(Line2D([0], [0], marker='.', ls = "", color=f'C{i}', label=f"{cell_name}"))

    ax.legend(handles=legend_points)



def plot_scatter_squares(df: pd.DataFrame, ax: Axes) -> None:
    for lab in sorted([lab for lab in df["label"].unique()]):
        ax.scatter(
            df[df["label"]==lab]["square_x"],
            df[df["label"]==lab]["square_y"],
            marker='s',
            s=20,
            label = f"lab: {lab}"
        )

    ax.legend()

In [ ]:
mapping = build_mapping(f"tsv/IF1_phen_to_cell_mapping.csv")

In [15]:
directory = f"tsv/cells_properties/"

for filename in os.listdir(directory):
    print(f"[{filename}] starting...")

    df_tmp_raw = pd.read_csv(os.path.join(directory, filename), sep="\t", compression="gzip")
    
    df_tmp_squares = analise_patient(df_tmp_raw, mapping, 250)

    df_tmp_squares.to_csv(f"outputs/{filename}.csv")


    df_tmp_squares["label"] = fast_hdbscan.HDBSCAN(
        min_cluster_size=32,
        min_samples=4,
    ).fit_predict(np.ascontiguousarray(df_tmp_squares.drop(columns="total square_x square_y".split(" ")).to_numpy(), dtype=np.float32))

    print(f"[{filename}] plot making...")
    fig, axs = plt.subplots(1, 2, figsize = (22, 10))

    plot_scatter_col(df_tmp_raw, axs[0])

    plot_scatter_squares(df_tmp_squares, axs[1])

    fig.savefig(f"outputs/{filename}.png")

    plt.close(fig)

[LUNG-NSCLC2-0613-FIXT-01-IF1-01_#_cells_properties_#_53ec91fc9c63f3088eb688a53912ac53.tsv.gz] starting...
[LUNG-NSCLC2-0613-FIXT-01-IF1-01_#_cells_properties_#_53ec91fc9c63f3088eb688a53912ac53.tsv.gz] plot making...


/tmp/ipykernel_47451/409232061.py:25: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(f"outputs/{filename}.png")


[LUNG-NSCLC2-0664-FIXT-01-IF1-01_#_cells_properties_#_e2995fa6cfa5cfdbd51cc83e8e465177.tsv.gz] starting...
[LUNG-NSCLC2-0664-FIXT-01-IF1-01_#_cells_properties_#_e2995fa6cfa5cfdbd51cc83e8e465177.tsv.gz] plot making...


/tmp/ipykernel_47451/409232061.py:25: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(f"outputs/{filename}.png")


[LUNG-NSCLC2-0695-FIXT-01-IF1-01_#_cells_properties_#_8f2bfb1ba17fa457f1fb5b8650a45cb1.tsv.gz] starting...
[LUNG-NSCLC2-0695-FIXT-01-IF1-01_#_cells_properties_#_8f2bfb1ba17fa457f1fb5b8650a45cb1.tsv.gz] plot making...
[LUNG-NSCLC2-0696-FIXT-01-IF1-01_#_cells_properties_#_9f8d9e704403e21802101969e9adef0d.tsv.gz] starting...
[LUNG-NSCLC2-0696-FIXT-01-IF1-01_#_cells_properties_#_9f8d9e704403e21802101969e9adef0d.tsv.gz] plot making...


...

/tmp/ipykernel_47451/409232061.py:25: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(f"outputs/{filename}.png")


[LUNG-NSCLC2-0587-FIXT-01-IF1-01_#_cells_properties_#_fb439474f2166fe87b6d3087643b2af3.tsv.gz] starting...
[LUNG-NSCLC2-0587-FIXT-01-IF1-01_#_cells_properties_#_fb439474f2166fe87b6d3087643b2af3.tsv.gz] plot making...


/tmp/ipykernel_47451/409232061.py:25: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(f"outputs/{filename}.png")


[LUNG-NSCLC2-0591-FIXT-01-IF1-01_#_cells_properties_#_b17f2afaad57821d4888ff6ba7f4b179.tsv.gz] starting...
[LUNG-NSCLC2-0591-FIXT-01-IF1-01_#_cells_properties_#_b17f2afaad57821d4888ff6ba7f4b179.tsv.gz] plot making...
[LUNG-NSCLC2-0659-FIXT-01-IF1-01_#_cells_properties_#_1a6b8562910080e4a4e518975b98dad9.tsv.gz] starting...
[LUNG-NSCLC2-0659-FIXT-01-IF1-01_#_cells_properties_#_1a6b8562910080e4a4e518975b98dad9.tsv.gz] plot making...
[LUNG-NSCLC2-0721-FIXT-01-IF1-01_#_cells_properties_#_b610ff2e25d9d911b0658bbb48284525.tsv.gz] starting...
[LUNG-NSCLC2-0721-FIXT-01-IF1-01_#_cells_properties_#_b610ff2e25d9d911b0658bbb48284525.tsv.gz] plot making...
[LUNG-NSCLC2-0565-FIXT-01-IF1-01_#_cells_properties_#_12cf4d92f8de6204055505affdf85e90.tsv.gz] starting...
[LUNG-NSCLC2-0565-FIXT-01-IF1-01_#_cells_properties_#_12cf4d92f8de6204055505affdf85e90.tsv.gz] plot making...
[LUNG-NSCLC2-0616-FIXT-01-IF1-01_#_cells_properties_#_99f9f40cba303b42b5a251859509f000.tsv.gz] starting...
[LUNG-NSCLC2-0616-FIXT-01

/tmp/ipykernel_47451/409232061.py:25: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(f"outputs/{filename}.png")
